# 01 — Data Preparation & Feature Engineering
**NeoStats DS Hackathon — Loan Application Refusal Prediction**

**Goal of this notebook:** turn the raw files into one clean, leakage-free, model-ready table.

| Step | What we do | Hackathon requirement |
|---|---|---|
| 1 | Load data, check shapes, types, missing values, duplicates | Part 1 |
| 2 | Build the target `REFUSED`, report the refusal rate | Part 1 |
| 3 | Join client details on `SK_ID_CURR`, verify join cardinality | Part 1 |
| 4 | Fix odd values (`365243`, `XNA`, `XAP`) | Part 1 |
| 5 | Remove post-decision (leakage) columns, with evidence | Part 1 |
| 6 | Convert `DAYS_*` to years | Part 1 |
| 7 | Engineer features (incl. point-in-time history) | Part 2 |
| 8 | Train / validation / test / OOT split | Section 3 |
| 9 | Save model table + dashboard tables | Parts 1, 3 |

> Encoding and imputation are **not** done here. They are fitted on the training set only, inside the sklearn pipeline in notebook 03 — otherwise information from validation/test/OOT would leak into preprocessing.

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

SEED = 42
np.random.seed(SEED)

# ---- Paths: works on Kaggle AND on your laptop (VS Code) without edits ----
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_DIRS = [Path(os.environ["DATA_DIR"])] if "DATA_DIR" in os.environ else []
SEARCH_DIRS += [Path("/kaggle/input")] if ON_KAGGLE else [Path("../data"), Path("data")]
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_file(name):
    # Return the path of name.csv (or name.zip) anywhere under the search folders.
    for d in SEARCH_DIRS:
        if not d.exists():
            continue
        for ext in (".csv", ".zip"):
            hits = sorted(d.rglob(f"{name}{ext}"))
            if hits:
                return hits[0]
    raise FileNotFoundError(f"{name}.csv/.zip not found under {SEARCH_DIRS}")

FILES = {n: find_file(n) for n in ["previous_application", "application_train", "application_test"]}
for k, v in FILES.items():
    print(f"{k:22s} -> {v}")
print("Outputs will be written to:", OUT_DIR.resolve())

## 1. Load `previous_application` (the main table) and profile it
One row = one loan application and the lender's decision. Text columns are stored as `category` to cut memory roughly in half.

In [2]:
def to_category(df):
    for c in df.columns:
        if df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype("category")
    return df

pa = to_category(pd.read_csv(FILES["previous_application"]))
print("Shape:", pa.shape)
print("Memory (MB):", round(pa.memory_usage(deep=True).sum() / 1e6))
print("Duplicate SK_ID_PREV:", pa["SK_ID_PREV"].duplicated().sum())
print("Fully duplicated rows:", pa.duplicated().sum())
print("\nColumn types:\n", pa.dtypes.astype(str).value_counts())

Shape: (1670214, 37)
Memory (MB): 307
Duplicate SK_ID_PREV: 0


Fully duplicated rows: 0

Column types:
 category    16
float64     15
int64        6
Name: count, dtype: int64


In [3]:
# Missing values in the raw table (top 20)
raw_missing = (pa.isna().mean() * 100).round(1).sort_values(ascending=False)
raw_missing.head(20).to_frame("% missing")

,% missing
RATE_INTEREST_PRIVILEGED,99.6
RATE_INTEREST_PRIMARY,99.6
AMT_DOWN_PAYMENT,53.6
RATE_DOWN_PAYMENT,53.6
NAME_TYPE_SUITE,49.1
DAYS_TERMINATION,40.3
DAYS_FIRST_DRAWING,40.3
DAYS_FIRST_DUE,40.3
DAYS_LAST_DUE_1ST_VERSION,40.3
DAYS_LAST_DUE,40.3


## 2. Build the target `REFUSED`
* **Approved → 0**, **Refused → 1**.
* **Canceled** and **Unused offer** are removed: in those cases the *customer* decided the outcome, not the lender. Our model must learn the lender's decision, so those rows would add noise to the target.

In [4]:
print(pa["NAME_CONTRACT_STATUS"].value_counts(), "\n")

# --- Point-in-time history features need ALL applications (incl. Canceled), so compute them before filtering ---
hist = pa[["SK_ID_CURR", "DAYS_DECISION", "NAME_CONTRACT_STATUS"]].copy()
hist["is_refused"] = (hist["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
day = (hist.groupby(["SK_ID_CURR", "DAYS_DECISION"], observed=True)
           .agg(n_apps=("is_refused", "size"), n_ref=("is_refused", "sum"))
           .reset_index()
           .sort_values(["SK_ID_CURR", "DAYS_DECISION"]))
g = day.groupby("SK_ID_CURR")
# cumulative count MINUS the current day => only applications decided strictly EARLIER
day["PRIOR_APP_COUNT"] = g["n_apps"].cumsum() - day["n_apps"]
day["PRIOR_REFUSED_COUNT"] = g["n_ref"].cumsum() - day["n_ref"]
day["DAYS_SINCE_PRIOR_APP"] = day["DAYS_DECISION"] - g["DAYS_DECISION"].shift(1)
day = day.drop(columns=["n_apps", "n_ref"])
del hist

df = pa[pa["NAME_CONTRACT_STATUS"].isin(["Approved", "Refused"])].copy()
del pa
df["REFUSED"] = (df["NAME_CONTRACT_STATUS"] == "Refused").astype("int8")
df["NAME_CONTRACT_STATUS"] = df["NAME_CONTRACT_STATUS"].cat.remove_unused_categories()

rate = df["REFUSED"].mean()
print(f"Rows after removing Canceled / Unused offer: {len(df):,}")
print(f"Refusal rate: {rate:.2%}   ->  about 1 refusal for every {(1 - rate) / rate:.1f} approvals")
print(f"A model that always says 'Approved' would get {1 - rate:.1%} accuracy and catch 0 refusals -> accuracy is misleading.")

NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64 



Rows after removing Canceled / Unused offer: 1,327,459
Refusal rate: 21.90%   ->  about 1 refusal for every 3.6 approvals
A model that always says 'Approved' would get 78.1% accuracy and catch 0 refusals -> accuracy is misleading.


**Class imbalance comment:** refusals are the minority class. This is *moderate* imbalance (not extreme like fraud), so we do **not** need SMOTE. In notebook 03 we use **class weights** (`class_weight="balanced"` for Logistic Regression, `scale_pos_weight` for LightGBM) and tune the **decision threshold** on validation. We judge models on PR-AUC, recall, precision and F1 — not accuracy.

## 3. Join client details on `SK_ID_CURR`
`application_train` + `application_test` together hold one row per client (same columns; test just has no `TARGET`). Using both gives client details for **every** application instead of ~85%. `TARGET` (a *future* loan-default outcome) is never loaded.

**Point-in-time rule:** the client tables describe the client at their *latest* application. Columns about that latest loan (`AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE`, `NAME_CONTRACT_TYPE`, …) and counts measured at that moment (credit-bureau enquiries, social-circle defaults, phone change) describe the *future* relative to older applications, so we do not load them. We load only stable client attributes.

We also leave `CODE_GENDER` out of the model features (kept for the dashboard only): using gender in credit decisions is a fairness and regulatory risk.

In [5]:
CLIENT_COLS = ["SK_ID_CURR", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY", "CNT_CHILDREN",
               "CNT_FAM_MEMBERS", "AMT_INCOME_TOTAL", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
               "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE",
               "REGION_POPULATION_RELATIVE", "REGION_RATING_CLIENT", "DAYS_BIRTH", "DAYS_EMPLOYED",
               "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

ct = pd.read_csv(FILES["application_train"], usecols=CLIENT_COLS).assign(CLIENT_SOURCE="train")
cs = pd.read_csv(FILES["application_test"], usecols=CLIENT_COLS).assign(CLIENT_SOURCE="test")
client = to_category(pd.concat([ct, cs], ignore_index=True))
del ct, cs

print("Client rows:", len(client), "| duplicate SK_ID_CURR:", client["SK_ID_CURR"].duplicated().sum())
assert client["SK_ID_CURR"].is_unique, "client table must have one row per SK_ID_CURR"

n_before = len(df)
df = df.merge(client, on="SK_ID_CURR", how="left", validate="many_to_one", indicator=True)
assert len(df) == n_before, "join changed the number of rows!"
print(f"Rows before/after join: {n_before:,} / {len(df):,}")
print("Join result:\n", df["_merge"].value_counts().to_string())
print("Refusal rate by client source:\n", df.groupby("CLIENT_SOURCE", observed=True)["REFUSED"].agg(["size", "mean"]).round(4))
df = df.drop(columns="_merge")

Client rows: 356255 | duplicate SK_ID_CURR: 0


Rows before/after join: 1,327,459 / 1,327,459
Join result:
 _merge
both          1327459
left_only           0
right_only          0
Refusal rate by client source:
                   size    mean
CLIENT_SOURCE                 
test            195970  0.2311
train          1131489  0.2169


## 4. Treat odd values
| Value | Meaning | Action |
|---|---|---|
| `365243` in `DAYS_*` | placeholder for "no date" | → missing (`NaN`). In `DAYS_EMPLOYED` it marks pensioners/unemployed, so we also keep a flag |
| `XNA` | not available | → missing (`NaN`) |
| `XAP` | not applicable (question did not apply, e.g. loan purpose for a POS loan) | → its own category `"Not applicable"` — it carries real meaning |

In [6]:
# 365243 placeholder
df["FLAG_NO_EMPLOYMENT_DATE"] = (df["DAYS_EMPLOYED"] == 365243).astype("int8")
for c in [c for c in df.columns if c.startswith("DAYS_")]:
    n = (df[c] == 365243).sum()
    if n:
        print(f"{c}: {n:,} placeholder values -> NaN")
        df.loc[df[c] == 365243, c] = np.nan

# XNA / XAP in text columns
cat_cols = [c for c in df.columns if isinstance(df[c].dtype, pd.CategoricalDtype)]
xna_xap = []
for c in cat_cols:
    n_xna, n_xap = int((df[c] == "XNA").sum()), int((df[c] == "XAP").sum())
    if n_xna or n_xap:
        xna_xap.append((c, n_xna, n_xap))
        s = df[c].astype("object").replace({"XNA": np.nan, "XAP": "Not applicable"})
        df[c] = s.astype("category")
pd.DataFrame(xna_xap, columns=["column", "XNA -> NaN", "XAP -> 'Not applicable'"])

DAYS_FIRST_DRAWING: 934,444 placeholder values -> NaN
DAYS_FIRST_DUE: 40,645 placeholder values -> NaN
DAYS_LAST_DUE_1ST_VERSION: 93,864 placeholder values -> NaN
DAYS_LAST_DUE: 211,221 placeholder values -> NaN
DAYS_TERMINATION: 225,913 placeholder values -> NaN
DAYS_EMPLOYED: 254,167 placeholder values -> NaN


,column,XNA -> NaN,XAP -> 'Not applicable'
0,NAME_CONTRACT_TYPE,31,0
1,NAME_CASH_LOAN_PURPOSE,410677,848991
2,NAME_PAYMENT_TYPE,313796,0
3,CODE_REJECT_REASON,5244,1036774
4,NAME_CLIENT_TYPE,932,0
5,NAME_GOODS_CATEGORY,634198,0
6,NAME_PORTFOLIO,40901,0
7,NAME_PRODUCT_TYPE,731209,0
8,NAME_SELLER_INDUSTRY,541351,0
9,NAME_YIELD_GROUP,185411,0


## 5. Remove data-leakage columns (with evidence)
A column leaks if it is only filled in **after** the lender decides. The test: *would I know this value when a brand-new application arrives?* The table below shows how each suspect column behaves for approved vs refused applications — a column that is missing for ~100% of refusals is recorded only when a loan is approved.

In [7]:
def leak_evidence(col):
    miss = df[col].isna()
    return pd.Series({
        "% missing | Approved": round(miss[df.REFUSED == 0].mean() * 100, 1),
        "% missing | Refused": round(miss[df.REFUSED == 1].mean() * 100, 1),
        "refusal rate when missing": round(df.loc[miss, "REFUSED"].mean() * 100, 1) if miss.any() else np.nan,
    })

suspects = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE",
            "DAYS_TERMINATION", "NFLAG_INSURED_ON_APPROVAL", "AMT_ANNUITY", "CNT_PAYMENT"]
display(pd.DataFrame({c: leak_evidence(c) for c in suspects}).T)

print("CODE_REJECT_REASON vs REFUSED:")
display(pd.crosstab(df["CODE_REJECT_REASON"].astype(str), df["REFUSED"]))
print("Share of applications with AMT_CREDIT = 0:", df.groupby("REFUSED")["AMT_CREDIT"].apply(lambda s: round((s == 0).mean(), 3)).to_dict())
for c in ["FLAG_LAST_APPL_PER_CONTRACT", "NFLAG_LAST_APPL_IN_DAY"]:
    print(f"\nRefusal rate by {c}:\n", df.groupby(c, observed=True)["REFUSED"].agg(["size", "mean"]).round(3))

,% missing | Approved,% missing | Refused,refusal rate when missing
DAYS_FIRST_DRAWING,94.0,100.0,23.0
DAYS_FIRST_DUE,7.7,100.0,78.4
DAYS_LAST_DUE_1ST_VERSION,12.9,100.0,68.5
DAYS_LAST_DUE,24.2,100.0,53.7
DAYS_TERMINATION,25.6,100.0,52.3
NFLAG_INSURED_ON_APPROVAL,3.8,100.0,88.0
AMT_ANNUITY,0.0,14.1,100.0
CNT_PAYMENT,0.0,14.1,100.0


CODE_REJECT_REASON vs REFUSED:


REFUSED,0,1
CODE_REJECT_REASON,,
HC,0,175231
LIMIT,0,55680
Not applicable,1036773,1
SCO,0,37467
SCOFR,0,12811
SYSTEM,0,717
VERIF,0,3535


Share of applications with AMT_CREDIT = 0: {0: 0.001, 1: 0.1}

Refusal rate by FLAG_LAST_APPL_PER_CONTRACT:
                                 size   mean
FLAG_LAST_APPL_PER_CONTRACT                
N                               8473  1.000
Y                            1318986  0.214

Refusal rate by NFLAG_LAST_APPL_IN_DAY:
                            size   mean
NFLAG_LAST_APPL_IN_DAY                
0                          5887  0.875
1                       1321572  0.216


In [8]:
LEAKAGE = {
    "NAME_CONTRACT_STATUS": "Raw decision itself - the target is built from it",
    "CODE_REJECT_REASON": "Reason for refusal - only exists once the lender has refused ('XAP' for all approvals)",
    "AMT_CREDIT": "FINAL amount granted by the lender; 0 for ~10% of refusals. Use AMT_APPLICATION (amount asked) instead",
    "AMT_ANNUITY": "Annuity of the offered loan; missing ONLY for refusals (100% refusal when missing) -> set during processing",
    "CNT_PAYMENT": "Term of the offered loan; same missing pattern as AMT_ANNUITY (100% refusal when missing)",
    "DAYS_FIRST_DRAWING": "Date money was paid out - exists only for approved loans",
    "DAYS_FIRST_DUE": "First repayment due date - exists only for approved loans",
    "DAYS_LAST_DUE_1ST_VERSION": "Repayment schedule date - exists only for approved loans",
    "DAYS_LAST_DUE": "Last due date - exists only for approved loans",
    "DAYS_TERMINATION": "Contract end date - exists only for approved loans",
    "NFLAG_INSURED_ON_APPROVAL": "Insurance requested *on approval* - missing for 100% of refusals",
    "FLAG_LAST_APPL_PER_CONTRACT": "Set afterwards when the client re-applies for the same contract ('N' rows are 100% refused)",
    "NFLAG_LAST_APPL_IN_DAY": "Whether a LATER application happened the same day - future information (0 rows ~87% refused)",
}
OTHER_DROPS = {
    "RATE_INTEREST_PRIMARY": "99.6% missing - no usable information",
    "RATE_INTEREST_PRIVILEGED": "99.6% missing - no usable information",
    "SELLERPLACE_AREA": "Seller-area ID code (~2,100 values) with no numeric meaning",
}
df = df.drop(columns=list(LEAKAGE) + list(OTHER_DROPS))
leak_log = pd.DataFrame(
    [(c, "Leakage (post-decision)", r) for c, r in LEAKAGE.items()] +
    [(c, "Data quality", r) for c, r in OTHER_DROPS.items()],
    columns=["column", "type", "reason"])
leak_log.to_csv(OUT_DIR / "removed_columns_log.csv", index=False)
print("Columns remaining:", df.shape[1])
leak_log

Columns remaining: 43


,column,type,reason
0,NAME_CONTRACT_STATUS,Leakage (post-decision),Raw decision itself - the target is built from it
1,CODE_REJECT_REASON,Leakage (post-decision),Reason for refusal - only exists once the lend...
2,AMT_CREDIT,Leakage (post-decision),FINAL amount granted by the lender; 0 for ~10%...
3,AMT_ANNUITY,Leakage (post-decision),Annuity of the offered loan; missing ONLY for ...
4,CNT_PAYMENT,Leakage (post-decision),Term of the offered loan; same missing pattern...
5,DAYS_FIRST_DRAWING,Leakage (post-decision),Date money was paid out - exists only for appr...
6,DAYS_FIRST_DUE,Leakage (post-decision),First repayment due date - exists only for app...
7,DAYS_LAST_DUE_1ST_VERSION,Leakage (post-decision),Repayment schedule date - exists only for appr...
8,DAYS_LAST_DUE,Leakage (post-decision),Last due date - exists only for approved loans
9,DAYS_TERMINATION,Leakage (post-decision),Contract end date - exists only for approved l...


## 6. Convert `DAYS_*` to years — **at the time of each application**
All `DAYS_*` values count back from the client's latest application. `DAYS_BIRTH` therefore gives age *today*, but an application decided 5 years ago was made by a younger person. So:

`age at decision = (DAYS_DECISION − DAYS_BIRTH) / 365.25`

Same for employment length. If the current job started *after* this application, employment length at that time is unknown → `NaN`.

In [9]:
df["AGE_AT_DECISION"] = ((df["DAYS_DECISION"] - df["DAYS_BIRTH"]) / 365.25).round(1)
emp = (df["DAYS_DECISION"] - df["DAYS_EMPLOYED"]) / 365.25
df["EMPLOYED_YEARS_AT_DECISION"] = emp.where(emp >= 0).round(1)
df["YEARS_BEFORE_LATEST"] = (-df["DAYS_DECISION"] / 365.25).round(2)
df["MONTHS_AGO"] = ((-df["DAYS_DECISION"] - 1) // 30 + 1).astype("int16")   # 1 = most recent month
df = df.drop(columns=["DAYS_BIRTH", "DAYS_EMPLOYED"])
df[["AGE_AT_DECISION", "EMPLOYED_YEARS_AT_DECISION", "YEARS_BEFORE_LATEST", "MONTHS_AGO"]].describe().round(1)

,AGE_AT_DECISION,EMPLOYED_YEARS_AT_DECISION,YEARS_BEFORE_LATEST,MONTHS_AGO
count,1327459.0,813559.0,1327459.0,1327459.0
mean,41.7,6.0,2.8,34.9
std,12.0,6.2,2.2,26.5
min,16.9,0.0,0.0,1.0
25%,31.8,1.7,1.1,13.0
50%,40.8,4.0,2.1,26.0
75%,51.7,8.0,4.2,52.0
max,68.9,48.2,8.0,98.0


## 7. Feature engineering
Each feature uses only information available when the application arrives. Fixed business cut-offs are used for bands (not quantiles of the full data), so nothing is "fitted" on validation/test/OOT.

| # | Feature | Business reason |
|---|---|---|
| 1 | `APP_TO_INCOME` | Amount asked vs annual income — affordability; big asks relative to income are riskier |
| 2 | `FINANCED_SHARE` | Amount asked / goods price — how much of the purchase is borrowed (>1 means borrowing more than the item costs) |
| 3 | `INCOME_PER_FAMILY_MEMBER` | Income stretched over dependants — disposable income proxy |
| 4 | `AGE_BAND` | Age at application, grouped — readable on the dashboard, captures non-linear age effects |
| 5 | `EMPLOYMENT_BAND` | Job tenure at application — stability of income |
| 6 | `INCOME_BAND` | Income grouped into business-friendly ranges |
| 7 | `IS_WEEKEND` | Weekend applications come through different channels/processes |
| 8 | `HOUR_BAND` | Night-time applications can indicate online/unusual behaviour |
| 9 | `EXT_SOURCE_MISSING_COUNT` | Clients with fewer external credit scores are "thin file" and harder to assess |
| 10 | `EXT_SOURCE_MEAN` | Single summary of the three external credit scores |
| 11 | `IS_ZERO_AMOUNT_REQUEST` | Applications with amount 0 behave differently (often card/limit requests) |
| 12 | `PRIOR_APP_COUNT` | How many applications the client made **before** this one |
| 13 | `PRIOR_REFUSED_COUNT` / `PRIOR_REFUSAL_RATE` | Past refusals — the lender's own history with this client |
| 14 | `DAYS_SINCE_PRIOR_APP` | Re-applying quickly (e.g. right after a refusal) is a strong signal |

Features 12–14 are the **stretch** point-in-time history features: computed only from applications with a strictly smaller (older) `DAYS_DECISION`, so no future information is used.

In [10]:
inc = df["AMT_INCOME_TOTAL"]
df["APP_TO_INCOME"] = df["AMT_APPLICATION"] / inc
df["FINANCED_SHARE"] = df["AMT_APPLICATION"] / df["AMT_GOODS_PRICE"].replace(0, np.nan)
df["INCOME_PER_FAMILY_MEMBER"] = inc / df["CNT_FAM_MEMBERS"].replace(0, np.nan)

df["AGE_BAND"] = pd.cut(df["AGE_AT_DECISION"], [0, 25, 35, 45, 55, 120],
                        labels=["<25", "25-34", "35-44", "45-54", "55+"], right=False)
df["EMPLOYMENT_BAND"] = pd.cut(df["EMPLOYED_YEARS_AT_DECISION"], [0, 1, 3, 5, 10, 100],
                               labels=["<1y", "1-3y", "3-5y", "5-10y", "10y+"], right=False)
df["EMPLOYMENT_BAND"] = (df["EMPLOYMENT_BAND"].cat.add_categories(["No employment date", "Unknown"])
                         .where(df["FLAG_NO_EMPLOYMENT_DATE"] == 0, "No employment date")
                         .fillna("Unknown"))
df["INCOME_BAND"] = pd.cut(inc, [0, 100_000, 150_000, 200_000, 300_000, np.inf],
                           labels=["<100k", "100-150k", "150-200k", "200-300k", "300k+"], right=False)

df["IS_WEEKEND"] = df["WEEKDAY_APPR_PROCESS_START"].isin(["SATURDAY", "SUNDAY"]).astype("int8")
df["HOUR_BAND"] = pd.cut(df["HOUR_APPR_PROCESS_START"], [0, 8, 12, 17, 24],
                         labels=["Night 0-7", "Morning 8-11", "Afternoon 12-16", "Evening 17-23"], right=False)

ext = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
df["EXT_SOURCE_MISSING_COUNT"] = ext.isna().sum(axis=1).astype("int8")
df["EXT_SOURCE_MEAN"] = ext.mean(axis=1)
df["IS_ZERO_AMOUNT_REQUEST"] = (df["AMT_APPLICATION"] == 0).astype("int8")

# point-in-time history (computed in step 2 from ALL applications, strictly earlier days only)
n_before = len(df)
df = df.merge(day, on=["SK_ID_CURR", "DAYS_DECISION"], how="left", validate="many_to_one")
assert len(df) == n_before
df["PRIOR_REFUSAL_RATE"] = df["PRIOR_REFUSED_COUNT"] / df["PRIOR_APP_COUNT"].replace(0, np.nan)
del day

ENGINEERED = ["APP_TO_INCOME", "FINANCED_SHARE", "INCOME_PER_FAMILY_MEMBER", "AGE_BAND", "EMPLOYMENT_BAND",
              "INCOME_BAND", "IS_WEEKEND", "HOUR_BAND", "EXT_SOURCE_MISSING_COUNT", "EXT_SOURCE_MEAN",
              "IS_ZERO_AMOUNT_REQUEST", "PRIOR_APP_COUNT", "PRIOR_REFUSED_COUNT", "PRIOR_REFUSAL_RATE",
              "DAYS_SINCE_PRIOR_APP"]
print("Engineered features:", len(ENGINEERED))
df[ENGINEERED].describe(include="all").T[["count", "mean", "min", "max", "unique", "top"]]

Engineered features: 15


,count,mean,min,max,unique,top
APP_TO_INCOME,1327459.0,1.361296,0.0,87.719298,NaN,NaN
FINANCED_SHARE,1241147.0,1.000008,0.0,9.0,NaN,NaN
INCOME_PER_FAMILY_MEMBER,1327459.0,95401.287222,2812.5,39000000.0,NaN,NaN
AGE_BAND,1327459,NaN,NaN,NaN,5,35-44
EMPLOYMENT_BAND,1327459,NaN,NaN,NaN,7,Unknown
INCOME_BAND,1327459,NaN,NaN,NaN,5,100-150k
IS_WEEKEND,1327459.0,0.25764,0.0,1.0,NaN,NaN
HOUR_BAND,1327459,NaN,NaN,NaN,4,Afternoon 12-16
EXT_SOURCE_MISSING_COUNT,1327459.0,0.710735,0.0,3.0,NaN,NaN
EXT_SOURCE_MEAN,1327197.0,0.502136,0.000006,0.863816,NaN,NaN


In [11]:
# Sanity check: refusal rate by number of prior refusals (a strong but legitimate history signal)
print(df.groupby(df["PRIOR_REFUSED_COUNT"].clip(upper=5))["REFUSED"].agg(["size", "mean"]).round(3))

                        size   mean
PRIOR_REFUSED_COUNT                
0                    1009768  0.154
1                     148886  0.333
2                      68392  0.419
3                      36373  0.495
4                      21012  0.545
5                      43028  0.648


## 8. Feature categorisation (deliverable)
Grouping the model features helps the EDA, the dashboard and the interview story.

In [12]:
FEATURE_GROUPS = {
    "Demographic": ["AGE_AT_DECISION", "AGE_BAND", "CNT_CHILDREN", "CNT_FAM_MEMBERS", "NAME_FAMILY_STATUS",
                    "NAME_EDUCATION_TYPE", "NAME_HOUSING_TYPE", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
                    "REGION_POPULATION_RELATIVE", "REGION_RATING_CLIENT"],
    "Financial": ["AMT_INCOME_TOTAL", "INCOME_BAND", "INCOME_PER_FAMILY_MEMBER", "NAME_INCOME_TYPE",
                  "OCCUPATION_TYPE", "ORGANIZATION_TYPE", "EMPLOYED_YEARS_AT_DECISION", "EMPLOYMENT_BAND",
                  "FLAG_NO_EMPLOYMENT_DATE", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
                  "EXT_SOURCE_MEAN", "EXT_SOURCE_MISSING_COUNT"],
    "Product": ["NAME_CONTRACT_TYPE", "AMT_APPLICATION", "AMT_GOODS_PRICE", "AMT_DOWN_PAYMENT",
                "RATE_DOWN_PAYMENT", "APP_TO_INCOME", "FINANCED_SHARE", "IS_ZERO_AMOUNT_REQUEST",
                "NAME_CASH_LOAN_PURPOSE", "NAME_GOODS_CATEGORY", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE",
                "NAME_YIELD_GROUP", "PRODUCT_COMBINATION", "NAME_PAYMENT_TYPE", "NAME_SELLER_INDUSTRY"],
    "Channel": ["CHANNEL_TYPE", "NAME_TYPE_SUITE", "WEEKDAY_APPR_PROCESS_START", "HOUR_APPR_PROCESS_START",
                "IS_WEEKEND", "HOUR_BAND"],
    "History": ["NAME_CLIENT_TYPE", "PRIOR_APP_COUNT", "PRIOR_REFUSED_COUNT", "PRIOR_REFUSAL_RATE",
                "DAYS_SINCE_PRIOR_APP"],
}
MODEL_FEATURES = [f for fs in FEATURE_GROUPS.values() for f in fs]
missing_feats = [f for f in MODEL_FEATURES if f not in df.columns]
assert not missing_feats, missing_feats
feature_groups = pd.DataFrame([(f, g) for g, fs in FEATURE_GROUPS.items() for f in fs], columns=["feature", "group"])
feature_groups.to_csv(OUT_DIR / "feature_groups.csv", index=False)
print("Model features:", len(MODEL_FEATURES))
print(feature_groups["group"].value_counts().to_string())

Model features: 52
group
Product        16
Financial      14
Demographic    11
Channel         6
History         5


## 9. Data-quality table (stretch deliverable)

In [13]:
def action_for(c):
    if c in ENGINEERED: return "Engineered feature"
    if c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]: return "Kept; missing imputed in pipeline (train-fitted) + missing-count feature"
    if df[c].isna().any():
        return "Kept; imputed in pipeline (median / 'Missing' category, fitted on train only)"
    return "Kept; no missing values"
dq = pd.DataFrame({
    "dtype": df[MODEL_FEATURES].dtypes.astype(str),
    "% missing": (df[MODEL_FEATURES].isna().mean() * 100).round(1),
    "n unique": df[MODEL_FEATURES].nunique(),
})
dq["action"] = [action_for(c) for c in dq.index]
dq = dq.sort_values("% missing", ascending=False)
dq.to_csv(OUT_DIR / "data_quality_table.csv")
dq.head(15)

,dtype,% missing,n unique,action
NAME_PRODUCT_TYPE,category,55.1,2,Kept; imputed in pipeline (median / 'Missing' ...
EXT_SOURCE_1,float64,54.1,129459,Kept; missing imputed in pipeline (train-fitte...
NAME_GOODS_CATEGORY,category,47.8,27,Kept; imputed in pipeline (median / 'Missing' ...
AMT_DOWN_PAYMENT,float64,43.5,29272,Kept; imputed in pipeline (median / 'Missing' ...
RATE_DOWN_PAYMENT,float64,43.5,207001,Kept; imputed in pipeline (median / 'Missing' ...
NAME_SELLER_INDUSTRY,category,40.8,10,Kept; imputed in pipeline (median / 'Missing' ...
EMPLOYED_YEARS_AT_DECISION,float64,38.7,466,Kept; imputed in pipeline (median / 'Missing' ...
NAME_TYPE_SUITE,category,37.2,7,Kept; imputed in pipeline (median / 'Missing' ...
OCCUPATION_TYPE,category,32.2,18,Kept; imputed in pipeline (median / 'Missing' ...
NAME_CASH_LOAN_PURPOSE,category,30.9,24,Kept; imputed in pipeline (median / 'Missing' ...


## 10. Train / validation / test / out-of-time (OOT) split
1. **OOT** = most recent 12 months: `DAYS_DECISION > -365` — kept aside to test the model on "future" data.
2. **Development** = older data (`DAYS_DECISION <= -365`) → stratified **70 / 15 / 15** train / validation / test (`random_state=42`).

Stratifying keeps the refusal rate identical across train, validation and test.

In [14]:
assert df["DAYS_DECISION"].notna().all(), "DAYS_DECISION has missing values"
is_oot = df["DAYS_DECISION"] > -365
dev_idx = df.index[~is_oot]

train_idx, temp_idx = train_test_split(dev_idx, test_size=0.30, stratify=df.loc[dev_idx, "REFUSED"], random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=df.loc[temp_idx, "REFUSED"], random_state=SEED)

df["SPLIT"] = "oot"
df.loc[train_idx, "SPLIT"] = "train"
df.loc[val_idx, "SPLIT"] = "validation"
df.loc[test_idx, "SPLIT"] = "test"

split_summary = (df.groupby("SPLIT")
                   .agg(rows=("REFUSED", "size"), refusal_rate=("REFUSED", "mean"),
                        days_decision_min=("DAYS_DECISION", "min"), days_decision_max=("DAYS_DECISION", "max"))
                   .reindex(["train", "validation", "test", "oot"]))
split_summary["share_of_all"] = (split_summary["rows"] / len(df)).round(3)
split_summary["refusal_rate"] = split_summary["refusal_rate"].round(4)
assert split_summary["rows"].sum() == len(df)
split_summary

,rows,refusal_rate,days_decision_min,days_decision_max,share_of_all
SPLIT,,,,,
train,714467,0.1728,-2922,-365,0.538
validation,153100,0.1728,-2922,-365,0.115
test,153101,0.1728,-2922,-365,0.115
oot,306791,0.3726,-364,-1,0.231


**Read this table carefully.** Train / validation / test have the same refusal rate (stratification worked) and cover the same older period. The OOT refusal rate is very different — the most recent year has a much higher share of refusals. That is **temporal drift** (a change in the base rate). Expect ranking metrics (ROC-AUC) to hold up better than threshold metrics (precision/recall) on OOT; we explain this in notebook 03.

*Caveat for the README:* `DAYS_DECISION` is relative to each client's own latest application, not a shared calendar, so "most recent 12 months" is approximate calendar time.

## 11. Save outputs
* `model_table.parquet` — every model feature + target + split (input to notebooks 02 and 03)
* `fact_applications.csv` — one row per application for the Power BI fact table
* `dim_client.csv` — one row per client for the Power BI client dimension

In [15]:
KEYS = ["SK_ID_PREV", "SK_ID_CURR", "DAYS_DECISION", "MONTHS_AGO", "YEARS_BEFORE_LATEST", "SPLIT", "REFUSED"]
model_table = df[KEYS + MODEL_FEATURES + ["CODE_GENDER"]]
model_table.to_parquet(OUT_DIR / "model_table.parquet", index=False)

FACT_COLS = ["SK_ID_PREV", "SK_ID_CURR", "SPLIT", "REFUSED", "DAYS_DECISION", "MONTHS_AGO",
             "NAME_CONTRACT_TYPE", "NAME_PRODUCT_TYPE", "NAME_PORTFOLIO", "CHANNEL_TYPE",
             "NAME_CASH_LOAN_PURPOSE", "NAME_CLIENT_TYPE", "NAME_GOODS_CATEGORY", "NAME_YIELD_GROUP",
             "AMT_APPLICATION", "AMT_GOODS_PRICE", "APP_TO_INCOME", "AGE_BAND", "EMPLOYMENT_BAND",
             "WEEKDAY_APPR_PROCESS_START", "IS_WEEKEND", "HOUR_BAND", "PRIOR_APP_COUNT", "PRIOR_REFUSED_COUNT"]
fact = df[FACT_COLS].copy()
fact["DECISION_STATUS"] = np.where(fact["REFUSED"] == 1, "Refused", "Approved")
fact.to_csv(OUT_DIR / "fact_applications.csv", index=False)

DIM_CLIENT_COLS = ["SK_ID_CURR", "CLIENT_SOURCE", "CODE_GENDER", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
                   "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "ORGANIZATION_TYPE",
                   "AMT_INCOME_TOTAL", "INCOME_BAND", "CNT_CHILDREN", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
                   "REGION_RATING_CLIENT", "EXT_SOURCE_MEAN", "EXT_SOURCE_MISSING_COUNT"]
dim_client = df[DIM_CLIENT_COLS].drop_duplicates("SK_ID_CURR")
assert dim_client["SK_ID_CURR"].is_unique and set(fact["SK_ID_CURR"]) <= set(dim_client["SK_ID_CURR"])
dim_client.to_csv(OUT_DIR / "dim_client.csv", index=False)

print("model_table      :", model_table.shape)
print("fact_applications:", fact.shape)
print("dim_client       :", dim_client.shape)
for f in sorted(OUT_DIR.glob("*")):
    if f.is_file():
        print(f"  {f.name:28s} {f.stat().st_size / 1e6:8.1f} MB")

model_table      : (1327459, 60)
fact_applications: (1327459, 25)
dim_client       : (338402, 17)
  data_quality_table.csv            0.0 MB
  dim_client.csv                   50.2 MB
  fact_applications.csv           241.6 MB
  feature_groups.csv                0.0 MB
  model_table.parquet              84.0 MB
  removed_columns_log.csv           0.0 MB


## Summary — what to say about this notebook
* **Target:** Approved = 0, Refused = 1, Canceled / Unused offer removed (customer-driven outcomes). Refusal rate printed in step 2.
* **Join:** many applications → one client, validated (`many_to_one`), row count unchanged; train + test client files give full coverage.
* **Leakage:** 13 post-decision columns removed with evidence (`removed_columns_log.csv`), plus client "latest-loan" columns never loaded.
* **Odd values:** 365243 → NaN (+ flag), XNA → NaN, XAP → "Not applicable".
* **Time-correct features:** age and job tenure measured at each decision; history features use strictly earlier applications only.
* **Splits:** OOT = last 12 months; older data stratified 70/15/15. Strong drift in refusal rate in OOT.
* **No imputation/encoding here** — fitted on train only in notebook 03.